<a href="https://colab.research.google.com/github/janithcyapa/DHCA-Framework/blob/main/Control%20Test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Control Test


In [ ]:
# @title Env

!pip install -q "energy-plus-utility @ git+https://github.com/janithcyapa/energy-plus-utility.git@main"
import importlib.metadata
ver = importlib.metadata.version("energy-plus-utility")
print(f"\n✅ Installed 'energy-plus-utility' version: {ver}")

# %%
from eplus import prepare_colab_eplus
prepare_colab_eplus(silent=False)

# %%
# Optional: install control (not used in this notebook but kept for compatibility)
!pip install -q control

# ## 2. Load Model (IDF + Weather)


import types, datetime, requests, io, os, gc
from pathlib import Path
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import traceback
from eplus.core import EPlusUtil

In [ ]:
# @title Setup
OUT_DIR = "/simulation/eplus_out"
url_idf = "https://raw.githubusercontent.com/janithcyapa/DHCA-Framework/refs/heads/main/System%20Models/PythonPluginUserDefined5ZoneAirCooled.idf"
url_epw = "https://raw.githubusercontent.com/janithcyapa/DHCA-Framework/refs/heads/main/System%20Models/Weather%20Files/LKA_Colombo-Katunayake.434500_SWERA.epw"

sim = EPlusUtil(verbose=3, out_dir=OUT_DIR)
sim.reset_state()
sim.delete_out_dir()
sim.clear_eplus_outputs(patterns="eplusout.*")
sim.set_model_from_url(url_idf, url_epw)


In [ ]:
# @title logger
specs = [
    {"name": "Site Outdoor Air Drybulb Temperature", "key": "Environment"},
    {"name": "Zone Mean Air Temperature", "key": "*"},
    {"name": "System Node Mass Flow Rate", "key": "SPACE1-1 IN NODE"},
    {"name": "System Node Mass Flow Rate", "key": "SPACE2-1 IN NODE"},
    {"name": "System Node Mass Flow Rate", "key": "SPACE3-1 IN NODE"},
    {"name": "System Node Mass Flow Rate", "key": "SPACE4-1 IN NODE"},
    {"name": "System Node Mass Flow Rate", "key": "SPACE5-1 IN NODE"},
    {"name": "System Node Mass Flow Rate", "key": "MAIN COOLING COIL 1 AIR OUTLET NODE"},
]
sim.ensure_output_variables(specs, activate=True)
sim.collected_data = []

def state_logger(self, state):
    try:
        if not self.api.exchange.api_data_fully_ready(state):
            return

        day = self.api.exchange.day_of_year(state)
        time_now = self.api.exchange.current_time(state)
        hours, mins = divmod(int(time_now * 60), 60)

        row = {
            "day": day,
            "hour": hours,
            "minute": mins,
            "time_decimal": time_now
        }

        def get_val(name, key):
            handle = self.api.exchange.get_variable_handle(state, name, key)
            return self.api.exchange.get_variable_value(state, handle) if handle != -1 else np.nan

        row["T_out"] = get_val("Site Outdoor Air Drybulb Temperature", "Environment")

        zones = ["SPACE1-1", "SPACE2-1", "SPACE3-1", "SPACE4-1", "SPACE5-1"]
        for z in zones:
            row[f"{z}_T_in"] = get_val("Zone Mean Air Temperature", z)
            row[f"{z}_Mdot"] = get_val("System Node Mass Flow Rate", f"{z} IN NODE")

        row["AHU_Mdot"] = get_val("System Node Mass Flow Rate", "MAIN COOLING COIL 1 AIR OUTLET NODE")

        # Add setpoints stored by controller
        row.update(getattr(self, 'control_setpoints', {}))
        self.collected_data.append(row)

    except Exception as e:
        print("\n!!! ERROR IN LOGGER !!!", file=sys.stderr)
        traceback.print_exc()

sim.state_logger = types.MethodType(state_logger, sim)

In [ ]:
# @title Control
def god_mode_control(self, state):
    try:
        if not self.api.exchange.api_data_fully_ready(state) or self.api.exchange.warmup_flag(state):
            return

        # --- Setpoint Definitions (kg/s) ---
        zone_flows = {
            "SPACE1-1 IN NODE": 0.4,
            "SPACE2-1 IN NODE": 0.3,
            "SPACE3-1 IN NODE": 0.2,
            "SPACE4-1 IN NODE": 0.3,
            "SPACE5-1 IN NODE": 0.5
        }
        ahu_total_flow = sum(zone_flows.values())

        # --- Apply to E+ Actuators ---
        for node_name, mdot in zone_flows.items():
            h = self.api.exchange.get_actuator_handle(state, "System Node Setpoint", "Mass Flow Rate", node_name)
            if h != -1:
                self.api.exchange.set_actuator_value(state, h, mdot)

        h_ahu = self.api.exchange.get_actuator_handle(state, "System Node Setpoint", "Mass Flow Rate", "MAIN COOLING COIL 1 AIR OUTLET NODE")
        if h_ahu != -1:
            self.api.exchange.set_actuator_value(state, h_ahu, ahu_total_flow)

        # --- Log the applied setpoints ---
        self.control_setpoints = {f"set_{k}": v for k, v in zone_flows.items()}
        self.control_setpoints["set_AHU_Mdot"] = ahu_total_flow

    except Exception as e:
        print("\n!!! ERROR IN CONTROLLER !!!", file=sys.stderr)
        traceback.print_exc()

sim.god_mode_control = types.MethodType(god_mode_control, sim)

In [ ]:
# @title Sim and Plot
sim.register_handlers("begin", [{"method_name": "state_logger"}])
sim.register_handlers("inside_iter", [{"method_name": "god_mode_control"}])

sim.set_simulation_params(
    start=(1, 1),
    end=(1, 2),          # Run for 1 day
    timestep_per_hour=4, # 15-minute steps
)

print("Starting simulation...")
res = sim.run_annual()

if res == 0:
    print("Simulation complete! Processing DataFrame...")
    df = pd.DataFrame(sim.collected_data)
    sim_start = pd.Timestamp("2026-01-01 00:00:00")
    df['datetime'] = sim_start + pd.to_timedelta(df['day']-1, unit='D') + pd.to_timedelta(df['hour'] + df['minute']/60, unit='h')
    df.set_index('datetime', inplace=True)
else:
    print("Simulation Failed! Check eplusout.err for standard E+ errors.")

# =========================================================
# 6. PLOT RESULTS
# =========================================================
def plot_results(df):
    dark_template = 'plotly_dark'
    zones = ["SPACE1-1", "SPACE2-1", "SPACE3-1", "SPACE4-1", "SPACE5-1"]

    # --- Plot 1: Temperatures ---
    fig1 = go.Figure(layout=dict(template=dark_template, title="Temperatures", yaxis_title="°C"))
    fig1.add_trace(go.Scatter(x=df.index, y=df['T_out'], name='Outdoor T', line=dict(dash='dash')))
    for z in zones:
        if f'{z}_T_in' in df.columns:
            fig1.add_trace(go.Scatter(x=df.index, y=df[f'{z}_T_in'], name=f'{z} Temp'))
    fig1.show()

    # --- Plot 2: Air Mass Flow Rates ---
    fig2 = go.Figure(layout=dict(template=dark_template, title="Air Mass Flow Rates", yaxis_title="kg/s"))
    for z in zones:
        if f'{z}_Mdot' in df.columns:
            fig2.add_trace(go.Scatter(x=df.index, y=df[f'{z}_Mdot'], name=f'{z} Actual Flow'))
            # Add the target setpoint as a dashed line
            target_col = f'set_{z} IN NODE'
            if target_col in df.columns:
                target = df[target_col].iloc[0]
                fig2.add_hline(y=target, line_dash="dot", annotation_text=f"{z} Setpoint")

    if 'AHU_Mdot' in df.columns:
        fig2.add_trace(go.Scatter(x=df.index, y=df['AHU_Mdot'], name='AHU Total Flow', line=dict(width=3)))
    fig2.show()

if res == 0:
    plot_results(df)
